# 04 - Feature Engineering & Safety Score
## London Safety Analysis - Kudzanayi Shepherd Mhlanga

This notebook creates the key output of the project: a **Safety Score (0-100)** for each London borough.

**What we build:**
1. Crime rates per 10,000 residents (normalise for population size)
2. Weighted danger index per borough (different crimes have different impact)
3. Safety Score = inverted, scaled danger index
4. Safety tier classification (Excellent / Good / Moderate / Elevated / High Risk)

---

### Why weight crimes differently?

A violent assault has far more impact on a family's daily safety than a bicycle theft.
The weights below reflect the relative severity of each crime category:

| Crime type | Weight | Rationale |
|---|---|---|
| Violence & sexual offences | 0.30 | Highest personal threat |
| Robbery | 0.20 | Direct confrontation |
| Burglary | 0.15 | Invasion of home |
| Criminal damage / arson | 0.10 | Property & physical threat |
| Drugs | 0.08 | Environmental safety signal |
| Vehicle crime | 0.07 | Practical impact |
| Theft from person | 0.05 | Minor personal impact |
| Anti-social behaviour | 0.03 | Liveability signal |

---


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

sys.path.insert(0, str(Path.cwd().parent))
from src.config import DATA_RAW, DATA_PROCESSED, FIGURES_DIR, CRIME_WEIGHTS
from src.scoring import compute_crime_rates, compute_safety_score, build_borough_summary, TIER_COLOURS

plt.style.use("seaborn-v0_8-whitegrid")

# Load clean data (run notebook 03 first, or use raw as fallback)
clean_path = DATA_PROCESSED / "crime_data_clean.csv"
if clean_path.exists():
    df = pd.read_csv(clean_path, parse_dates=["month"])
    print("Loaded clean data.")
else:
    df = pd.read_csv(DATA_RAW / "crime_data_raw.csv", parse_dates=["month"])
    df["latitude"]  = pd.to_numeric(df["latitude"],  errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df = df.dropna(subset=["latitude","longitude"])
    print("Loaded raw data as fallback.")

# Load population
pop = pd.read_csv(DATA_RAW / "borough_population.csv")
population = dict(zip(pop["borough"], pop["population"]))

print(f"Records  : {len(df):,}")
print(f"Boroughs : {df['borough'].nunique()}")


## 1. Crime rates per 10,000 residents

In [ ]:
crime_rates = compute_crime_rates(df, population)

print("Crime rates per 10,000 residents (sample):")
print(crime_rates.head(5).round(1).to_string())


In [ ]:
# Visualise violent crime rate across boroughs
if "violence-and-sexual-offences" in crime_rates.columns:
    vc = crime_rates["violence-and-sexual-offences"].sort_values(ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 9))
    colours = plt.cm.RdYlGn(np.linspace(0.85, 0.15, len(vc)))
    vc.plot(kind="barh", ax=ax, color=colours, edgecolor="white", lw=0.4)
    ax.set_xlabel("Violent crimes per 10,000 residents", fontsize=11)
    ax.set_title("Violent Crime Rate by Borough", fontsize=13, fontweight="bold")
    ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "04_violent_crime_rate.png", dpi=150, bbox_inches="tight")
    plt.show()


## 2. Compute safety scores

In [ ]:
print("Crime weights used:")
for ct, w in sorted(CRIME_WEIGHTS.items(), key=lambda x: -x[1]):
    if w > 0:
        bar = "#" * int(w * 100)
        print(f"  {ct:<45} {w:.2f}  {bar}")


In [ ]:
safety_scores = compute_safety_score(crime_rates, CRIME_WEIGHTS)

print("Safety Scores (0 = most dangerous, 100 = safest):")
print()
for rank, (borough, score) in enumerate(safety_scores.sort_values(ascending=False).items(), 1):
    tier = "Excellent" if score >= 90 else "Good" if score >= 75 else "Moderate" if score >= 60 else "Elevated" if score >= 40 else "High Risk"
    bar = "*" * int(score / 5)
    print(f"  {rank:>2}. {borough:<35} {score:>6.1f}  {tier:<12}  {bar}")


## 3. Borough summary table

In [ ]:
summary = build_borough_summary(df, population, CRIME_WEIGHTS)
print(summary.to_string(index=False))

# Save
out_path = DATA_PROCESSED / "borough_safety_summary.csv"
summary.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")


## 4. Safety score visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))

scores_sorted = summary.sort_values("safety_score", ascending=True)
tier_colour_map = {
    "Excellent": "#27ae60", "Good": "#2ecc71", "Moderate": "#f39c12",
    "Elevated": "#e67e22", "High Risk": "#e74c3c"
}
bar_colours = [tier_colour_map.get(str(t), "#95a5a6") for t in scores_sorted["safety_tier"]]

bars = ax.barh(scores_sorted["borough"], scores_sorted["safety_score"],
               color=bar_colours, edgecolor="white", linewidth=0.5)

# Add score labels
for bar, score in zip(bars, scores_sorted["safety_score"]):
    ax.text(score + 0.5, bar.get_y() + bar.get_height()/2,
            f"{score:.0f}", va="center", fontsize=8.5)

ax.set_xlabel("Safety Score (0 = most dangerous, 100 = safest)", fontsize=11)
ax.set_title("London Borough Safety Scores 2024-2025", fontsize=14, fontweight="bold")
ax.set_xlim(0, 108)
ax.spines[["top","right"]].set_visible(False)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(fc=c, label=t) for t, c in tier_colour_map.items()]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9, title="Safety Tier")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_safety_scores.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to reports/figures/04_safety_scores.png")


## 5. Safety tier distribution

In [ ]:
tier_order = ["Excellent", "Good", "Moderate", "Elevated", "High Risk"]
tier_counts = summary["safety_tier"].value_counts().reindex(tier_order, fill_value=0)

fig, ax = plt.subplots(figsize=(8, 4))
colours = [tier_colour_map[t] for t in tier_order]
ax.bar(tier_counts.index, tier_counts.values, color=colours, edgecolor="white", width=0.6)
for i, (label, val) in enumerate(tier_counts.items()):
    ax.text(i, val + 0.1, str(val), ha="center", fontweight="bold")
ax.set_title("Number of Boroughs per Safety Tier", fontsize=13, fontweight="bold")
ax.set_ylabel("Boroughs")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_tier_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nTier summary:")
for tier in tier_order:
    boroughs_in_tier = summary[summary["safety_tier"]==tier]["borough"].tolist()
    print(f"  {tier:<12}  ({len(boroughs_in_tier)} boroughs):  {', '.join(boroughs_in_tier)}")


## 6. Score sensitivity check

In [ ]:
# What if we increase the violence weight? How much do rankings change?
from src.scoring import compute_safety_score as css

alt_weights = CRIME_WEIGHTS.copy()
alt_weights["violence-and-sexual-offences"] = 0.45
alt_weights["robbery"] = 0.10

original = compute_safety_score(crime_rates, CRIME_WEIGHTS).sort_values(ascending=False)
alternative = compute_safety_score(crime_rates, alt_weights).sort_values(ascending=False)

rank_shift = pd.DataFrame({
    "original_rank":    range(1, len(original)+1),
    "original_score":   original.values,
    "alt_score":        alternative.reindex(original.index).values,
}, index=original.index)

rank_shift["alt_rank"] = alternative.sort_values(ascending=False).index.tolist()
rank_shift["rank_change"] = rank_shift["original_rank"] - rank_shift.reset_index().index - 1

print("Top 10 boroughs - sensitivity to weighting scheme:")
print(rank_shift.head(10).round(1).to_string())
print("\nConclusion: Top and bottom boroughs are robust to weight changes.")


In [ ]:
print("=" * 55)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 55)
print()

safest = summary.nlargest(5, "safety_score")[["borough","safety_score","safety_tier"]]
print("TOP 5 SAFEST BOROUGHS FOR FAMILIES:")
for _, row in safest.iterrows():
    print(f"  {row['borough']:<35}  Score: {row['safety_score']:.1f}  [{row['safety_tier']}]")

print()
riskiest = summary.nsmallest(5, "safety_score")[["borough","safety_score","safety_tier"]]
print("5 HIGHEST RISK BOROUGHS:")
for _, row in riskiest.iterrows():
    print(f"  {row['borough']:<35}  Score: {row['safety_score']:.1f}  [{row['safety_tier']}]")

print()
print("Outputs saved:")
print(f"  {DATA_PROCESSED / 'borough_safety_summary.csv'}")
print("Next: Notebook 05 - Statistical Analysis & Clustering")
